# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [1]:
%pip install librosa
%pip install panns-inference

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install torchcodec

Note: you may need to restart the kernel to use updated packages.


# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [ ]:
from datasets import load_dataset

# load the dataset from huggingface model hub
data = load_dataset("ashraq/esc50", split="train")
data

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

In [ ]:
# select the audio feature and display top three
audios = data["audio"]
audios[:3]

We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [ ]:
import numpy as np

# select only the audio data from the dataset and store in a numpy array
audios = np.array([a["array"] for a in data["audio"]])

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [7]:
from panns_inference import AudioTagging

# load the default model into the gpu.
model = AudioTagging(checkpoint_path=None, device='mps') # change device to cpu if a gpu is not available

Checkpoint path: /Users/franromlob/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [8]:
from dotenv import load_dotenv, find_dotenv
import os
env_path = find_dotenv()
print('dotenv path:', env_path)
_ = load_dotenv(env_path)

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
# initialize connection to pinecone (get API key at app.pinecone.io)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or "YOUR_API_KEY"
print('PINECONE_API_KEY loaded:', bool(PINECONE_API_KEY) and PINECONE_API_KEY != 'YOUR_API_KEY')

dotenv path: /Users/franromlob/Documents/IRONHACK/LAB/LangChain & RAG - Advanced RAG & Evaluation/lab-music-recommendations/.env
PINECONE_API_KEY loaded: True


In [13]:
import os
from pinecone import Pinecone

# configure client
pc = Pinecone(api_key=PINECONE_API_KEY)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [14]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Create the index:

In [15]:
index_name = "audio-search-demo"

In [16]:
import time

# check if index already exists (it shouldn't if this is first time)
index_response = pc.list_indexes()
print('Pinecone list_indexes response:', index_response)
if hasattr(index_response, 'names'):
    index_names = index_response.names()
elif isinstance(index_response, dict):
    index_names = [idx['name'] for idx in index_response.get('indexes', [])]
else:
    index_names = list(index_response)
print('Pinecone index names:', index_names)

if index_name not in index_names:
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=2048,
        metric='cosine',
        spec= spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
if index is None:
    raise RuntimeError('Pinecone index connection failed: pc.Index(index_name) returned None')
# view index stats
print('index describe stats:', index.describe_index_stats())

Pinecone list_indexes response: {'indexes': [{'dimension': 384,
              'host': 'question-answering-jocajog.svc.aped-4627-b74a.pinecone.io',
              'metric': 'cosine',
              'name': 'question-answering',
              'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
              'status': {'ready': True, 'state': 'Ready'}}]}
Pinecone index names: ['question-answering']
index describe stats: {'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}


# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [17]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(audios))
    # extract batch
    batch = audios[i:i_end]
    # generate embeddings for all the audios in the batch
    _, emb = model.inference(batch)
    # create unique IDs
    ids = [f"{idx}" for idx in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb.tolist()))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

100%|██████████| 32/32 [03:47<00:00,  7.11s/it]


{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 2000}},
 'total_vector_count': 2000}

We now have *2000* audio records indexed in Pinecone, we're ready to begin querying.

# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [ ]:
from IPython.display import Audio, display

# we set an audio number to select from the dataset
audio_num = 400
# get the audio data of the audio number
query_audio = data[audio_num]["audio"]["array"]
# get the category of the audio number
category = data[audio_num]["category"]
# print the category and play the audio
print("Query Audio:", category)
Audio(query_audio, rate=44100)

We have got the sound of a car horn. Let's generate an embedding for this sound.

In [19]:
# reshape query audio
query_audio = query_audio[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)
xq.shape

(1, 2048)

We have now converted the audio into a 2048-dimension vector the same way we did for all the other audio we indexed. Let's use this to query our Pinecone index.

In [20]:
# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)
results

{'matches': [{'id': '400', 'score': 0.999630928, 'values': []},
             {'id': '1667', 'score': 0.842086077, 'values': []},
             {'id': '1666', 'score': 0.831267, 'values': []}],
 'namespace': '',
 'usage': {'read_units': 1}}

Notice that the top result is the audio number 400 from our dataset, which is our query audio (the most similar item should always be the query itself). Let's listen to the top three results. 

In [ ]:
# play the top 3 similar audios
for r in results["matches"]:
    # select the audio data from the database using the id as an index
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

We have great results, everything aligns with what seems to be a busy city street with car horns.

Let's write a helper function to run the queries using audio from our dataset easily. We do not need to embed these audio samples again as we have already, they are just stored in Pinecone. So, we specify the `id` of the query audio to search with and tell Pinecone to search with that.

In [ ]:
def find_similar_audios(id):
    print("Query Audio:")
    # select the audio data from the database using the id as an index
    query_audio = data[id]["audio"]["array"]
    # play the query audio
    display(Audio(query_audio, rate=44100))
    # query pinecone index with the query audio id
    result = index.query(id=str(id), top_k=5)
    print("Result:")
    # play the top 5 similar audios
    for r in result["matches"]:
        a = data[int(r["id"])]["audio"]["array"]
        display(Audio(a, rate=44100))

In [30]:
find_similar_audios(1953)

Query Audio:


Result:


Here we return a set of revving motors (they seem to either be vehicles or lawnmowers).

In [31]:
find_similar_audios(452)

Query Audio:


Result:


And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [33]:
!wget https://storage.googleapis.com/audioset/miaow_16k.wav

--2026-06-06 16:08:16--  https://storage.googleapis.com/audioset/miaow_16k.wav
Resolving storage.googleapis.com (storage.googleapis.com)... 216.58.205.187, 172.217.20.251, 216.58.204.187, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|216.58.205.187|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 215546 (210K) [audio/x-wav]
Saving to: ‘miaow_16k.wav’

miaow_16k.wav       100%[===================>] 210.49K   500KB/s    in 0.4s    

2026-06-06 16:08:17 (500 KB/s) - ‘miaow_16k.wav’ saved [215546/215546]



We can load the audio into a Numpy array as follows:

In [34]:
import librosa

a, _ = librosa.load("./data/miaow_16k.wav", sr=44100)
Audio(a, rate=44100)

Now we generate the embeddings for this audio and query the Pinecone index.

In [ ]:
# reshape query audio
query_audio = a[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)

# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)

# play the top 3 similar audios
for r in results["matches"]:
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

Our audio search application has identified a set of similar cat sounds, which is excellent.

# Music Recommendation System

We now build our own recommendation system on top of the ESC-50 dataset.
The key difference from the demo above is that we create a **separate index**
called `music-recommendations` to keep our work isolated from the original
`audio-search-demo` index.

In [ ]:
import time

rec_index_name = "music-recommendations"

# check existing indexes
index_response = pc.list_indexes()
if isinstance(index_response, dict):
    index_names = [idx['name'] for idx in index_response.get('indexes', [])]
else:
    index_names = list(index_response)

if rec_index_name not in index_names:
    pc.create_index(
        rec_index_name,
        dimension=2048,
        metric='cosine',
        spec=spec
    )
    while not pc.describe_index(rec_index_name).status['ready']:
        time.sleep(1)

rec_index = pc.Index(rec_index_name)
print(rec_index.describe_index_stats())

## Upsert with Metadata

Unlike the original upsert, here we attach **metadata** to each vector:
the `category` and `filename` of each audio recording. This allows the
recommendation system to return not just similar sounds, but also human-readable
information about what was found — making results interpretable without
having to play every audio file.

Each record upserted to Pinecone now has this shape:
```
(id, embedding_vector, {"category": "dog", "filename": "1-100032-A-0.wav"})
```

In [ ]:
from tqdm.auto import tqdm

batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    i_end = min(i + batch_size, len(audios))
    batch = audios[i:i_end]
    _, emb = model.inference(batch)
    ids = [str(idx) for idx in range(i, i_end)]
    metadata = [
        {
            "category": data[idx]["category"],
            "filename": data[idx]["filename"]
        }
        for idx in range(i, i_end)
    ]
    to_upsert = list(zip(ids, emb.tolist(), metadata))
    rec_index.upsert(vectors=to_upsert)

print(rec_index.describe_index_stats())

## Recommendation Function

We define a `recommend()` function that:
1. Takes an `audio_id` from the dataset as input
2. Displays the query audio with its category and filename
3. Queries Pinecone using the stored vector ID (no re-embedding needed)
4. Returns the top-K most similar sounds with their similarity score,
   category and filename — and plays each one

The parameter `include_metadata=True` in the query is what enables
Pinecone to return the category and filename we stored during upsert.

In [ ]:
from IPython.display import Audio, display

def recommend(audio_id: int, top_k: int = 5) -> None:
    query_audio = data[audio_id]["audio"]["array"]
    category = data[audio_id]["category"]
    print(f"Query: [{category}] - {data[audio_id]['filename']}")
    display(Audio(query_audio, rate=44100))

    results = rec_index.query(id=str(audio_id), top_k=top_k, include_metadata=True)
    print(f"\nTop {top_k} recommendations:")
    for r in results["matches"]:
        meta = r.get("metadata", {})
        print(f"  score={r['score']:.3f} | category={meta.get('category')} | file={meta.get('filename')}")
        a = data[int(r["id"])]["audio"]["array"]
        display(Audio(a, rate=44100))

## Testing the Recommendation System

We test the system with three different sound categories from ESC-50
to verify that recommendations are acoustically coherent:
- `audio_id=400` → car horn (urban/traffic sounds)
- `audio_id=452` → birds (nature sounds)
- `audio_id=1642` → engine/motor (mechanical sounds)

A good recommendation system should return sounds from the same or
acoustically similar categories.

In [ ]:
recommend(400)   # car_horn
recommend(452)   # birds
recommend(1642)  # engine/motor

# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [36]:
pc.delete_index(index_name)